In [17]:
# GSPO demo: two prompt groups, batch size = 1 prompt group
# Group Sequence Policy Optimization
# 1) Sequence Level Loss
# 2) Rho clip is normalized by seq_len
# 3) No reference policy / no KL
# ============================================================

import copy
import numpy as np
import torch
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModelForCausalLM

torch.manual_seed(0)
np.random.seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "sshleifer/tiny-gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

/Users/skomirishett/opt/miniconda3/envs/meraki/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [18]:
# ------------------------------------------------------------
# 0) Load policy
# ------------------------------------------------------------

base_policy = AutoModelForCausalLM.from_pretrained(model_name).to(device)

# Current trainable policy p
policy_new = copy.deepcopy(base_policy).to(device)

# For demo clarity, keep dropout off.
# Gradients still flow because requires_grad=True.
policy_new.eval()
for p in policy_new.parameters():
    p.requires_grad_(True)

print("Loaded:", model_name, "on", device)

Loaded: sshleifer/tiny-gpt2 on cpu


In [19]:
def encode_prompt_and_response(tokenizer, prompt_text, completion_text):
    """
    Returns:
      input_ids: [1, seq_len] for prompt+completion (teacher-forced)
      prompt_len: #tokens in prompt
      comp_len:   #tokens in completion
    """
    prompt_ids = tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False)["input_ids"]
    full_ids   = tokenizer(prompt_text + completion_text, return_tensors="pt", add_special_tokens=False)["input_ids"]
    prompt_len = prompt_ids.shape[1]
    comp_len   = full_ids.shape[1] - prompt_len
    assert comp_len > 0, "Completion must add at least one token."
    return full_ids, prompt_len, comp_len

In [20]:
# ------------------------------------------------------------
# 2) Per-token response log-probs
# ------------------------------------------------------------

def token_logprobs_for_response(model, input_ids: torch.Tensor, prompt_len: int):
    """
    Computes per-token log-probs for response tokens.

    Causal LM convention:
        logits[:, k, :] predicts token at position k + 1

    For response token at absolute position pos:
        probability comes from logits at pos - 1.

    Returns:
        response_token_ids: [T]
        response_logps:     [T]
    """
    model_device = next(model.parameters()).device
    input_ids = input_ids.to(model_device)

    out = model(input_ids, use_cache=False)

    logits = out.logits  # [1, seq_len, vocab]
    logp_all = F.log_softmax(logits, dim=-1)

    token_ids = input_ids[0]  # [seq_len]
    seq_len = token_ids.shape[0]

    response_token_ids = token_ids[prompt_len:]  # [T]

    prev_pos = torch.arange(
        prompt_len - 1,
        seq_len - 1,
        device=model_device,
    )  # [T]

    lp_rows = logp_all[0, prev_pos, :]  # [T, vocab]

    response_logps = lp_rows.gather(
        dim=1,
        index=response_token_ids.unsqueeze(1),
    ).squeeze(1)  # [T]

    return response_token_ids, response_logps

In [21]:
def per_token_logps(model, tokenizer, prompt: str, completion: str):
    """
    Teacher-force prompt + completion and score completion tokens.
    """
    input_ids, prompt_len, _ = encode_prompt_and_response(
        tokenizer,
        prompt,
        completion,
    )

    response_ids, response_logps = token_logprobs_for_response(
        model,
        input_ids,
        prompt_len,
    )

    return response_ids, response_logps

In [22]:
def sequence_logprob_sum(logps: torch.Tensor):
    """
    log pi(response | prompt) = sum_t log pi(token_t | prefix)
    """
    return logps.sum()

In [23]:
# ------------------------------------------------------------
# 3) Toy reward
# ------------------------------------------------------------

def toy_reward(prompt: str, completion: str) -> float:
    """
    Toy scalar reward per completion.

    In real GSPO, this comes from a verifier or reward model.
    """
    c = completion.lower()

    reward = 0.0

    if "cat is on the" in prompt:
        if "sleep" in c:
            reward += 2.0
        if "wall" in c:
            reward += 1.0
        if "floor" in c:
            reward += 0.5
        if "runs" in c:
            reward -= 1.0
        if "mat" in c:
            reward -= 0.5

    if "2 + 2" in prompt:
        if "4" in c or "four" in c:
            reward += 2.0
        if "5" in c:
            reward -= 1.0

    # Tiny length tie-breaker
    reward += 0.05 * len(c.split())

    return float(reward)

In [24]:
# ------------------------------------------------------------
# 4) GSPO math helpers
# ------------------------------------------------------------

def group_sequence_advantages(rewards: np.ndarray, eps: float = 1e-8):
    """
    Group-normalized sequence-level advantage:

        A_i = (r_i - mean(group_rewards)) / (std(group_rewards) + eps)

    One scalar advantage per completion.
    """
    mean_r = float(np.mean(rewards))
    std_r = float(np.std(rewards))

    advantages = (rewards - mean_r) / (std_r + eps)

    return advantages.astype(np.float64), mean_r, std_r

In [25]:
def gspo_sequence_ratio(
    logp_seq_new: torch.Tensor,
    logp_seq_old: torch.Tensor,
    seq_len: int,
):
    """
    GSPO sequence-level importance ratio:

        s_i = exp((logp_seq_new - logp_seq_old) / |y_i|)

    This is length-normalized sequence likelihood ratio.
    """
    return torch.exp(
        (logp_seq_new - logp_seq_old) / float(seq_len)
    )

In [26]:
def gspo_sequence_clipped_objective(
    sequence_ratio: torch.Tensor,
    advantage: torch.Tensor,
    eps_clip: float,
):
    """
    GSPO clipped sequence objective:

        min(
            s_i * A_i,
            clip(s_i, 1-eps, 1+eps) * A_i
        )
    """
    s_clip = torch.clamp(
        sequence_ratio,
        min=1.0 - eps_clip,
        max=1.0 + eps_clip,
    )

    surr1 = sequence_ratio * advantage
    surr2 = s_clip * advantage

    objective_i = torch.minimum(surr1, surr2)

    return objective_i, s_clip, surr1, surr2

In [27]:
# ------------------------------------------------------------
# 5) Collect rollout group
# ------------------------------------------------------------

@torch.no_grad()
def collect_gspo_rollout_group(
    policy_old,
    tokenizer,
    prompt: str,
    completions: list[str],
    rewards: np.ndarray,
):
    """
    Rollout-time GSPO quantities.

    For one prompt group:
        - completion tokens
        - old per-token log-probs
        - old sequence log-prob
        - group-normalized sequence-level advantage
    """
    advantages, mean_r, std_r = group_sequence_advantages(rewards)

    rollout_items = []

    for i, completion in enumerate(completions):
        token_ids_old, logps_old = per_token_logps(
            policy_old,
            tokenizer,
            prompt,
            completion,
        )

        seq_len = int(logps_old.numel())

        logp_seq_old = sequence_logprob_sum(logps_old)

        rollout_items.append(
            {
                "completion": completion,
                "token_ids": token_ids_old.detach().cpu(),
                "logps_old": logps_old.detach().cpu(),
                "logp_seq_old": logp_seq_old.detach().cpu(),
                "seq_len": seq_len,
                "reward": float(rewards[i]),
                "advantage": float(advantages[i]),
            }
        )

    group_stats = {
        "mean_reward": mean_r,
        "std_reward": std_r,
        "advantages": advantages,
    }

    return rollout_items, group_stats


In [28]:
# ------------------------------------------------------------
# 6) GSPO objective computation
# ------------------------------------------------------------

def gspo_compute_objective(
    policy_new,
    tokenizer,
    prompt: str,
    rollout_items: list[dict],
    eps_clip: float,
):
    """
    Computes GSPO objective over one prompt group.

    For each completion i:
        logp_seq_new = sum_t log pi_new(y_i,t | prefix)
        logp_seq_old = stored rollout sequence log-prob

        s_i = exp((logp_seq_new - logp_seq_old) / T_i)

        objective_i = min(
            s_i * A_i,
            clip(s_i, 1-eps, 1+eps) * A_i
        )

    Group objective:
        J = mean_i objective_i

    Loss:
        loss = -J
    """
    per_sequence_objectives = []

    sequence_ratios = []

    for item in rollout_items:
        completion = item["completion"]

        token_ids_old = item["token_ids"].to(device)
        logp_seq_old = item["logp_seq_old"].to(device)
        seq_len = int(item["seq_len"])

        advantage = torch.tensor(
            item["advantage"],
            device=device,
            dtype=logp_seq_old.dtype,
        )

        token_ids_new, logps_new = per_token_logps(
            policy_new,
            tokenizer,
            prompt,
            completion,
        )

        assert torch.equal(token_ids_new, token_ids_old), (
            "Token mismatch between rollout tokens and current-policy scoring."
        )

        logp_seq_new = sequence_logprob_sum(logps_new)

        s_i = gspo_sequence_ratio(
            logp_seq_new=logp_seq_new,
            logp_seq_old=logp_seq_old,
            seq_len=seq_len,
        )

        objective_i, s_clip, surr1, surr2 = gspo_sequence_clipped_objective(
            sequence_ratio=s_i,
            advantage=advantage,
            eps_clip=eps_clip,
        )

        per_sequence_objectives.append(objective_i)
        sequence_ratios.append(s_i.detach())

    J = torch.stack(per_sequence_objectives).mean()

    loss = -J

    with torch.no_grad():
        ratio_tensor = torch.stack(sequence_ratios)

        clipfrac = (
            ((ratio_tensor > 1.0 + eps_clip) | (ratio_tensor < 1.0 - eps_clip))
            .float()
            .mean()
            .item()
        )

        stats = {
            "loss": float(loss.detach().cpu()),
            "objective": float(J.detach().cpu()),
            "mean_sequence_ratio": float(ratio_tensor.mean().cpu()),
            "min_sequence_ratio": float(ratio_tensor.min().cpu()),
            "max_sequence_ratio": float(ratio_tensor.max().cpu()),
            "clipfrac": clipfrac,
            "per_sequence_objective": (
                torch.stack(per_sequence_objectives).detach().cpu().numpy()
            ),
        }

    return loss, stats

In [29]:
# ------------------------------------------------------------
# 7) GSPO update step
# ------------------------------------------------------------

def gspo_update_step(
    policy_new,
    optimizer,
    tokenizer,
    prompt: str,
    rollout_items: list[dict],
    eps_clip: float,
):
    """
    One GSPO optimization step over one prompt group.
    """
    loss, stats = gspo_compute_objective(
        policy_new,
        tokenizer,
        prompt,
        rollout_items,
        eps_clip=eps_clip,
    )

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    return stats

In [30]:
# ------------------------------------------------------------
# 8) Evaluation helper
# ------------------------------------------------------------

@torch.no_grad()
def gspo_eval_stats(
    policy_new,
    tokenizer,
    prompt: str,
    rollout_items: list[dict],
    eps_clip: float,
):
    """
    Compute GSPO stats without updating.
    """
    loss, stats = gspo_compute_objective(
        policy_new,
        tokenizer,
        prompt,
        rollout_items,
        eps_clip=eps_clip,
    )

    return stats

In [31]:
 # ------------------------------------------------------------
# 9) GSPO settings
# ------------------------------------------------------------

num_gspo_iterations = 2

# GSPO ratio is sequence-length normalized, so smaller clip values
# are often reasonable. Use larger values for toy visibility.
eps_clip = 0.02

lr = 5e-4

optimizer = torch.optim.Adam(
    policy_new.parameters(),
    lr=lr,
)


# ------------------------------------------------------------
# 10) Two prompt groups, batch size = 1 prompt group
# ------------------------------------------------------------

examples = [
    {
        "prompt": "cat is on the",
        "completions": [
            " wall and its sleeping.",
            " mat and it runs.",
            " floor.",
        ],
    },
    {
        "prompt": "2 + 2 =",
        "completions": [
            " 4.",
            " 5.",
            " four.",
        ],
    },
]


# ------------------------------------------------------------
# 11) Main GSPO flow
# ------------------------------------------------------------

for example_idx, ex in enumerate(examples, start=1):

    print("\n" + "=" * 80)
    print(f"GSPO EXAMPLE {example_idx} / BATCH SIZE = 1 PROMPT GROUP")
    print("=" * 80)

    prompt = ex["prompt"]
    completions = ex["completions"]

    print("Prompt:", repr(prompt))
    print("Completions:")
    for completion in completions:
        print("   ", repr(completion))

    # --------------------------------------------------------
    # A) Create rollout snapshot q for this prompt group
    # --------------------------------------------------------
    policy_old = copy.deepcopy(policy_new).to(device).eval()
    for p in policy_old.parameters():
        p.requires_grad_(False)

    print("\n[1] Rollout snapshot")
    print("    q = policy_old = frozen copy of current policy_new")
    print("    q is fixed for this prompt group.")

    # --------------------------------------------------------
    # B) Compute rewards for the completion group
    # --------------------------------------------------------
    print("\n[2] Compute group rewards")

    rewards = np.array(
        [toy_reward(prompt, completion) for completion in completions],
        dtype=np.float64,
    )

    print("    rewards:", rewards)

    # --------------------------------------------------------
    # C) Collect rollout log-probs and sequence advantages
    # --------------------------------------------------------
    print("\n[3] Collect rollout log-probs and group sequence advantages")

    rollout_items, group_stats = collect_gspo_rollout_group(
        policy_old,
        tokenizer,
        prompt,
        completions,
        rewards,
    )

    print("    mean_reward:", group_stats["mean_reward"])
    print("    std_reward: ", group_stats["std_reward"])
    print("    advantages: ", group_stats["advantages"])

    for item in rollout_items:
        print(
            "    completion:",
            repr(item["completion"]),
            "reward:",
            item["reward"],
            "advantage:",
            item["advantage"],
            "seq_len:",
            item["seq_len"],
        )

    # --------------------------------------------------------
    # D) Before first GSPO update
    # --------------------------------------------------------
    print("\n[4] Before GSPO update")

    before = gspo_eval_stats(
        policy_new,
        tokenizer,
        prompt,
        rollout_items,
        eps_clip=eps_clip,
    )

    print("    sequence ratio mean/min/max:",
          before["mean_sequence_ratio"],
          before["min_sequence_ratio"],
          before["max_sequence_ratio"])
    print("    objective:", before["objective"])
    print("    loss:     ", before["loss"])
    print("    clipfrac: ", before["clipfrac"])

    # --------------------------------------------------------
    # E) GSPO update iterations on same rollout group
    # --------------------------------------------------------
    print("\n[5] GSPO update loop")
    print("    num_gspo_iterations:", num_gspo_iterations)
    print("    logp_seq_old/q stays fixed during these iterations.")
    print("    logp_seq_new/p is recomputed each iteration.")
    print("    eps_clip:", eps_clip)

    for gspo_iter in range(1, num_gspo_iterations + 1):

        stats = gspo_update_step(
            policy_new,
            optimizer,
            tokenizer,
            prompt,
            rollout_items,
            eps_clip=eps_clip,
        )

        print(f"\n    GSPO iteration {gspo_iter}")
        print("        sequence ratio mean/min/max:",
              stats["mean_sequence_ratio"],
              stats["min_sequence_ratio"],
              stats["max_sequence_ratio"])
        print("        objective:", stats["objective"])
        print("        loss:     ", stats["loss"])
        print("        clipfrac: ", stats["clipfrac"])
        print("        per_sequence_objective:",
              stats["per_sequence_objective"])

    # --------------------------------------------------------
    # F) After update, evaluate same rollout group
    # --------------------------------------------------------
    print("\n[6] After GSPO update on same rollout group")

    after = gspo_eval_stats(
        policy_new,
        tokenizer,
        prompt,
        rollout_items,
        eps_clip=eps_clip,
    )

    print("    sequence ratio mean/min/max:",
          after["mean_sequence_ratio"],
          after["min_sequence_ratio"],
          after["max_sequence_ratio"])
    print("    objective:", after["objective"])
    print("    loss:     ", after["loss"])
    print("    clipfrac: ", after["clipfrac"])

    print("\n[7] Discard this rollout group")
    print("    Next prompt group will create a fresh policy_old snapshot.")


GSPO EXAMPLE 1 / BATCH SIZE = 1 PROMPT GROUP
Prompt: 'cat is on the'
Completions:
    ' wall and its sleeping.'
    ' mat and it runs.'
    ' floor.'

[1] Rollout snapshot
    q = policy_old = frozen copy of current policy_new
    q is fixed for this prompt group.

[2] Compute group rewards
    rewards: [ 3.2  -1.3   0.55]

[3] Collect rollout log-probs and group sequence advantages
    mean_reward: 0.8166666666666668
    std_reward:  1.84676895023594
    advantages:  [ 1.29054223 -1.1461459  -0.14439633]
    completion: ' wall and its sleeping.' reward: 3.2 advantage: 1.290542230593286 seq_len: 5
    completion: ' mat and it runs.' reward: -1.3 advantage: -1.1461458971003309 seq_len: 5
    completion: ' floor.' reward: 0.55 advantage: -0.1443963334929551 seq_len: 2

[4] Before GSPO update
    sequence ratio mean/min/max: 1.0 1.0 1.0
    objective: -9.934107758624577e-09
    loss:      9.934107758624577e-09
    clipfrac:  0.0

[5] GSPO update loop
    num_gspo_iterations: 2
    logp_s